# 01 EDA: GTSRB
Run all cells, then **save the notebook with outputs** and commit it.

In [ ]:
import os, sys, numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
sys.path.append("..")
from src.utils import CLASS_NAMES
from src.dataset import add_track_group
ROOT = "/kaggle/input/gtsrb-german-traffic-sign"   # change if running elsewhere
train = add_track_group(pd.read_csv(os.path.join(ROOT, "Train.csv")))
test = pd.read_csv(os.path.join(ROOT, "Test.csv"))
print(len(train), "train images,", len(test), "test images,", train.ClassId.nunique(), "classes,", train.group.nunique(), "tracks")

## 1. Class distribution (imbalance)

In [ ]:
counts = train.ClassId.value_counts().sort_index()
plt.figure(figsize=(14,4)); plt.bar(counts.index, counts.values)
plt.xlabel("class id"); plt.ylabel("train images"); plt.title("Class distribution"); plt.show()
print("Max/min ratio:", counts.max()/counts.min())

## 2. One sample per class

In [ ]:
fig, axes = plt.subplots(5, 9, figsize=(16,10))
for ax in axes.ravel(): ax.axis("off")
for c in range(43):
    row = train[train.ClassId==c].iloc[0]
    img = cv2.cvtColor(cv2.imread(os.path.join(ROOT, row.Path)), cv2.COLOR_BGR2RGB)
    axes.ravel()[c].imshow(img); axes.ravel()[c].set_title(f"{c}: {CLASS_NAMES[c]}", fontsize=7)
plt.tight_layout(); plt.show()

## 3. Image sizes

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,3.5))
ax[0].hist(train.Width, bins=40); ax[0].set_title("Width (px)")
ax[1].hist(train.Height, bins=40); ax[1].set_title("Height (px)"); plt.show()
print(train[["Width","Height"]].describe())

## 4. Brightness distribution (random sample of 3000 images)

In [ ]:
sample = train.sample(3000, random_state=42)
bright = [cv2.imread(os.path.join(ROOT, p), cv2.IMREAD_GRAYSCALE).mean() for p in sample.Path]
plt.hist(bright, bins=40); plt.xlabel("mean grey level (0-255)"); plt.title("Brightness distribution"); plt.show()

## 5. Tracks: why we split by track

In [ ]:
tracks = train.groupby("ClassId").group.nunique()
print("Tracks per class: min", tracks.min(), "max", tracks.max())
print("Frames per track:", train.groupby("group").size().describe()[["min","mean","max"]].round(1).to_dict())

## Findings (write your own, 3-5 bullets)
- Imbalance: ...
- Image sizes: ...
- Brightness: ...
- Why track-split matters: ...